# Lab 04: Statistics Review II

**Week:** 2
**Type:** Independent
**Corresponding working sessions:** `working-sessions/statistics/`
**Estimated time:** 55–70 minutes
**Dataset:** Titanic — same dataset as Lab 03

---
### Learning Objectives
By the end of this lab you will be able to:
- Compute expected value and variance from a probability mass function
- Use the normal distribution to calculate probabilities and z-scores
- Explain why the Central Limit Theorem makes inference possible
- Build and interpret confidence intervals for a population mean

---
### A note before you start

This lab is independent — exercises give you less scaffolding than Lab 03.
The code structure is up to you. If you get stuck at any point, your working
session notebooks in `working-sessions/statistics/` are open for reference.
Use them. That is what they are there for.

You have the skills. Trust them.

Run the cells in order. Each section builds on the one before it.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key,
    make_grading_summary
)
from tkh_utils import load_titanic

In [2]:
_ak = make_answer_key({
    'q1': 'B',
    'q2': 'B',
    'q3': 'D',
    'q4': 'C',
})

---
## Dataset: Titanic

We are continuing with the Titanic dataset from Lab 03. Run the cells below
to reload and inspect it.

In [3]:
titanic = load_titanic()
print("Shape:", titanic.shape)
print()
titanic.head()

Shape: (891, 15)



,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
titanic.info()

**Reminder — key columns used in this lab:**

| Column | Type | Description |
|--------|------|-------------|
| `age` | float | Age in years (177 missing values — use `.dropna()`) |
| `fare` | float | Ticket fare in British pounds |
| `survived` | int | 1 = survived, 0 = did not survive |
| `pclass` | int | Passenger class (1, 2, or 3) |

---
## Section A — Conceptual Questions

For each question below, replace `"___"` with your answer (`"A"`, `"B"`, `"C"`, or `"D"`).
Run the cell after filling in your answer — you will get immediate feedback.

In [5]:
# Question 1
# A quality control engineer inspects a batch of 500 items.
# Each item independently has a 2% chance of being defective.
# Which distribution best models the number of defective items?
#
#   A) Poisson
#   B) Binomial
#   C) Normal
#   D) Uniform

q1_answer = "B"  # replace ___ with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer."
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite. There are a fixed number of trials (500) and each has the same probability (0.02). " \
    "Revisit working-sessions/statistics/10_discrete_distributions.ipynb"
print("\u2713 Question 1 correct!")

✓ Question 1 correct!


In [6]:
# Question 2
# In a normal distribution, approximately what percentage of values
# fall within two standard deviations of the mean?
#
#   A) 68%
#   B) 95%
#   C) 99.7%
#   D) 50%

q2_answer = "B"  # replace ___ with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer."
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite. Remember the 68-95-99.7 rule: 1 SD covers 68%, 2 SDs cover 95%, 3 SDs cover 99.7%. " \
    "Revisit working-sessions/statistics/12_normal_distribution.ipynb"
print("\u2713 Question 2 correct!")

✓ Question 2 correct!


In [7]:
# Question 3
# A population has standard deviation sigma = 20.
# A researcher increases the sample size from n = 25 to n = 100.
# What happens to the standard error?
#
#   A) It doubles, from 4 to 8
#   B) It stays at 4
#   C) It decreases to 1
#   D) It halves, from 4 to 2

q3_answer = "D"  # replace ___ with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer."
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite. SE = sigma / sqrt(n). Compute SE at n=25 and n=100 and compare. " \
    "Revisit working-sessions/statistics/14_sampling_distributions.ipynb"
print("\u2713 Question 3 correct!")

✓ Question 3 correct!


In [9]:
# Question 4
# A 95% confidence interval for mean passenger age is (28.6, 30.8).
# Which statement is the correct interpretation?
#
#   A) There is a 95% probability that the population mean is between 28.6 and 30.8
#   B) 95% of individual passengers have ages between 28.6 and 30.8
#   C) If we repeated this sampling procedure many times, approximately 95%
#      of the resulting intervals would contain the true population mean
#   D) The population mean is definitely between 28.6 and 30.8

q4_answer = "C"  # replace ___ with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer."
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite. The confidence level describes the long-run behavior of the procedure, " \
    "not the probability that this specific interval captures the mean. " \
    "Revisit working-sessions/statistics/16_confidence_intervals.ipynb"
print("\u2713 Question 4 correct!")

✓ Question 4 correct!


---
## Section B — Code Exercises

Fill in each `___` to complete the code. There may be more than one blank per exercise.
Run the cell to check your work.

In [10]:
# Exercise B1 — Expected value and variance from a PMF
# A soccer team's goals per match follow this distribution:
#
#   Goals (x):      0     1     2     3     4
#   Probability:  0.20  0.35  0.25  0.15  0.05
#
# Compute E[X] (expected number of goals) and Var[X] (variance).
# Recall: E[X] = sum(x * P(X=x))  and  Var[X] = sum((x - E[X])^2 * P(X=x))

x = np.array([0, 1, 2, 3, 4])
p = np.array([0.20, 0.35, 0.25, 0.15, 0.05])

expected_value = sum(x*p)   # E[X]
variance       = sum((x-expected_value)**2 * p)   # Var[X]

# --- check ---
assert abs(expected_value - 1.5) < 0.001, \
    ("E[X] should be 1.5 — check the formula: sum(x * p). "
     "Revisit working-sessions/statistics/09_discrete_random_variables.ipynb")
assert abs(variance - 1.25) < 0.001, \
    ("Var[X] should be 1.25 — check: sum((x - E[X])**2 * p). "
     "Revisit working-sessions/statistics/09_discrete_random_variables.ipynb")
print(f"\u2713 E[X] = {expected_value:.2f},  Var[X] = {variance:.2f}")

✓ E[X] = 1.50,  Var[X] = 1.25


In [12]:
# Exercise B2 — Normal distribution probabilities
# Exam scores are normally distributed with mean = 72 and standard deviation = 12.
# Use scipy.stats.norm to compute:
#   (1) the probability a randomly selected student scores below 60
#   (2) the probability a randomly selected student scores above 90
#
# Hint: norm.cdf(x, loc=mean, scale=std) gives P(X <= x)

p_below_60 = stats.norm.cdf(60, loc=72, scale=12)   # P(X < 60) — use stats.norm.cdf
p_above_90 = 1 - stats.norm.cdf(90, loc=72, scale=12)   # P(X > 90) — think about what 1 - cdf gives you

# --- check ---
assert abs(p_below_60 - 0.1587) < 0.001, \
    ("P(X < 60) should be near 0.159 — "
     "revisit working-sessions/statistics/12_normal_distribution.ipynb")
assert abs(p_above_90 - 0.0668) < 0.001, \
    ("P(X > 90) should be near 0.067 — remember to use 1 - CDF. "
     "Revisit working-sessions/statistics/12_normal_distribution.ipynb")
print(f"\u2713 P(score < 60) = {p_below_60:.4f}")
print(f"   P(score > 90) = {p_above_90:.4f}")

✓ P(score < 60) = 0.1587
   P(score > 90) = 0.0668


In [14]:
# Exercise B3 — Z-score standardization
# Using the same exam distribution (mean = 72, sigma = 12):
# Compute the z-score for a student who scored 96 and one who scored 60.
# Then compute the percentile for the student who scored 96.
#
# Recall: z = (x - mu) / sigma

mu    = 72
sigma = 12

z_score_96   = (96-mu)/sigma   # z-score for a score of 96
z_score_60   = (60-mu)/sigma   # z-score for a score of 60
percentile_96 = stats.norm.cdf(z_score_96)*100   # percentile for the student who scored 96 (0 to 100 scale)
                      # hint: stats.norm.cdf(z_score_96) * 100

# --- check ---
assert abs(z_score_96 - 2.0) < 0.001, \
    ("z-score for 96 should be 2.0 — "
     "revisit working-sessions/statistics/15_zscores_standardization.ipynb")
assert abs(z_score_60 - (-1.0)) < 0.001, \
    "z-score for 60 should be -1.0"
assert abs(percentile_96 - 97.72) < 0.1, \
    ("Percentile for z=2.0 should be near 97.7 — "
     "revisit working-sessions/statistics/15_zscores_standardization.ipynb")
print(f"\u2713 z(96) = {z_score_96:.1f}  -> {percentile_96:.1f}th percentile")
print(f"   z(60) = {z_score_60:.1f}")

✓ z(96) = 2.0  -> 97.7th percentile
   z(60) = -1.0


---
## Section C — Confidence Interval Analysis

Use the Titanic `age` column to build confidence intervals for the true mean
age of all passengers.

**Task 1:** Filter `titanic` to rows with non-null ages. Store the resulting
Series in `age_data`. Then compute:
- `n` — number of passengers with a known age
- `x_bar` — sample mean age
- `s` — sample standard deviation (use `ddof=1`)
- `se` — standard error of the mean: s / sqrt(n)

**Task 2:** Compute a 95% confidence interval. Store as `ci_95`, a tuple `(lower, upper)`.
Use z* = 1.96 for 95% confidence.

**Task 3:** Compute a 90% confidence interval. Store as `ci_90`, a tuple `(lower, upper)`.
Use z* = 1.645 for 90% confidence.

Recall: CI = x_bar ± z\* × se

In [18]:
# Task 1 — filter and compute sample statistics
age_data = titanic['age'].dropna()                # titanic['age'] with nulls dropped
n     = len(age_data)                   # number of rows
x_bar = age_data.mean()                   # sample mean
s     = age_data.std(ddof =1)                   # sample std (ddof=1)
se    = s/np.sqrt(n)                   # standard error: s / sqrt(n)

# Task 2 — 95% confidence interval  (z* = 1.96)
ci_95 = (x_bar-1.96*se, x_bar+1.96*se)            # (lower, upper)

# Task 3 — 90% confidence interval  (z* = 1.645)
ci_90 = (x_bar-1.645*se, x_bar+1.645*se)            # (lower, upper)


In [19]:
# --- Section C checks ---
assert isinstance(age_data, pd.Series), \
    "age_data should be a pandas Series — use titanic['age'].dropna()"
assert n == 714, \
    "n should be 714 — the number of passengers with a known age"
assert abs(x_bar - 29.70) < 0.5, \
    "x_bar should be near 29.7 — check .mean()"
assert 0.50 < se < 0.60, \
    ("SE should be near 0.54 — check s / sqrt(n). "
     "Revisit working-sessions/statistics/14_sampling_distributions.ipynb")

assert isinstance(ci_95, tuple) and len(ci_95) == 2, \
    "ci_95 should be a tuple (lower, upper)"
assert isinstance(ci_90, tuple) and len(ci_90) == 2, \
    "ci_90 should be a tuple (lower, upper)"

assert ci_95[0] < ci_90[0] < x_bar < ci_90[1] < ci_95[1], \
    ("The 90% CI should be narrower than the 95% CI — "
     "higher confidence requires a wider interval. "
     "Revisit working-sessions/statistics/16_confidence_intervals.ipynb")

assert abs((ci_95[1] - ci_95[0]) - 2 * 1.96 * se) < 0.01, \
    "Width of the 95% CI should equal 2 * 1.96 * SE"
assert abs((ci_90[1] - ci_90[0]) - 2 * 1.645 * se) < 0.01, \
    "Width of the 90% CI should equal 2 * 1.645 * SE"

print("\u2713 Section C complete!")
print()
print(f"Sample size:      n = {n}")
print(f"Sample mean:  x_bar = {x_bar:.2f} years")
print(f"Std deviation:    s = {s:.2f}")
print(f"Standard error:  SE = {se:.4f}")
print()
print(f"90% CI: ({ci_90[0]:.2f}, {ci_90[1]:.2f})")
print(f"95% CI: ({ci_95[0]:.2f}, {ci_95[1]:.2f})")

✓ Section C complete!

Sample size:      n = 714
Sample mean:  x_bar = 29.70 years
Std deviation:    s = 14.53
Standard error:  SE = 0.5436

90% CI: (28.80, 30.59)
95% CI: (28.63, 30.76)


In [ ]:
# --- Section C visualization ---
fig, ax = plt.subplots(figsize=(9, 3))

for label, ci, color in [("95% CI", ci_95, PALETTE[0]), ("90% CI", ci_90, PALETTE[1])]:
    ax.barh(label, ci[1] - ci[0], left=ci[0], height=0.4, color=color, alpha=0.75)

ax.axvline(x_bar, color="black", linestyle="--", linewidth=1.2, label=f"x̄ = {x_bar:.1f}")
ax.set_xlabel("Age (years)")
ax.set_title("Confidence Intervals for Mean Passenger Age")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()


---
## Section D — Interpretation

These questions are for reflection. Edit the markdown cells below each question
to write your response. There is no single right answer — we are looking for
thoughtful engagement with the concepts.

**Question D1:** Passenger age is not perfectly normally distributed — it has
some right skew and 177 missing values. Yet in Section C you used the normal
distribution to build confidence intervals. Why is this valid? What assumption
or theorem makes it acceptable to apply the normal-based formula here?

*Your response here...*

**Question D2:** Your 95% CI for mean passenger age is approximately (28.6, 30.8).
A classmate says: "That means there is a 95% chance the true mean age is between
28.6 and 30.8." Is this correct? Write out the right interpretation in your own
words, and explain what is wrong with your classmate's version.

*Your response here...*

**Question D3:** Suppose you had access to the full Titanic passenger manifest
(2,224 people) rather than just the 891 in this dataset. Assuming a similar
proportion of ages are available, how would the confidence interval change?
Would the center shift? Would the width change? Why?

*Your response here...*

---
## Self-Grading Summary

Run the cell below when you are ready to check your full score.
Fix any failing checks and re-run until everything passes.

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: Binomial vs Poisson"),
    (q2_answer, _ak['q2'], "Q2: 68-95-99.7 rule"),
    (q3_answer, _ak['q3'], "Q3: Standard error and sample size"),
    (q4_answer, _ak['q4'], "Q4: Correct CI interpretation"),
], total=4)
print("Section B & C: check \u2713 marks above")
print("Section D: for your own reflection")